# 📊 Análise da Camada Gold — Indicador de Alfabetização Brasil

Este notebook explora os datasets analíticos gerados pela camada Gold do pipeline.

**Datasets analisados:**
- `ranking_uf` — Ranking dos estados por taxa de alfabetização
- `meta_vs_realizado_uf` — Comparação entre meta e resultado
- `evolucao_uf` — Evolução 2023 → 2024
- `painel_nacional` — Visão consolidada nacional

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# Configurações visuais
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

GOLD_DIR = Path('..') / 'layers' / 'gold'

# Carrega os datasets
ranking     = pd.read_parquet(GOLD_DIR / 'ranking_uf.parquet')
meta_vs_rea = pd.read_parquet(GOLD_DIR / 'meta_vs_realizado_uf.parquet')
evolucao    = pd.read_parquet(GOLD_DIR / 'evolucao_uf.parquet')
painel      = pd.read_parquet(GOLD_DIR / 'painel_nacional.parquet')

print('✅ Datasets carregados com sucesso!')
print(f'  ranking_uf:          {len(ranking)} registros')
print(f'  meta_vs_realizado_uf:{len(meta_vs_rea)} registros')
print(f'  evolucao_uf:         {len(evolucao)} registros')
print(f'  painel_nacional:     {len(painel)} registros')

: 

## 1️⃣ Ranking dos Estados por Taxa de Alfabetização (2024)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 8))

cores = ['#2ecc71' if bateu else '#e74c3c' for bateu in ranking['bateu_meta_2024']]

bars = ax.barh(ranking['sigla_uf'], ranking['taxa_alfabetizacao'], color=cores)

# Linha da meta 2024
ax.axvline(x=80, color='navy', linestyle='--', linewidth=1.5, label='Meta 2030 (80%)')

# Valores nas barras
for bar, val in zip(bars, ranking['taxa_alfabetizacao']):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=9)

verde = mpatches.Patch(color='#2ecc71', label='Acima da meta 2024')
vermelho = mpatches.Patch(color='#e74c3c', label='Abaixo da meta 2024')
ax.legend(handles=[verde, vermelho, plt.Line2D([0],[0], color='navy', linestyle='--')],
          labels=['Acima da meta 2024', 'Abaixo da meta 2024', 'Meta 2030 (80%)'])

ax.set_xlabel('Taxa de Alfabetização (%)')
ax.set_title('Ranking dos Estados — Taxa de Alfabetização 2024', fontsize=14, fontweight='bold')
ax.set_xlim(0, 105)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 2️⃣ Meta vs Realizado por Estado (2024)

In [ ]:
df_2024 = meta_vs_rea[meta_vs_rea['ano'] == 2024].copy()

fig, ax = plt.subplots(figsize=(14, 7))

x = range(len(df_2024))
width = 0.35

bars1 = ax.bar([i - width/2 for i in x], df_2024['taxa_realizada'], width,
               label='Realizado', color='#3498db')
bars2 = ax.bar([i + width/2 for i in x], df_2024['meta_ano_vigente'], width,
               label='Meta 2024', color='#f39c12', alpha=0.8)

ax.set_xticks(list(x))
ax.set_xticklabels(df_2024['sigla_uf'], rotation=45, ha='right')
ax.set_ylabel('Taxa de Alfabetização (%)')
ax.set_title('Meta vs Realizado por Estado — 2024', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(y=80, color='red', linestyle='--', linewidth=1, label='Meta 2030')
plt.tight_layout()
plt.show()

## 3️⃣ Evolução 2023 → 2024 por Estado

In [ ]:
df_ev = evolucao.dropna(subset=['variacao_absoluta']).sort_values('variacao_absoluta', ascending=False)

fig, ax = plt.subplots(figsize=(14, 7))

cores = ['#2ecc71' if v >= 0 else '#e74c3c' for v in df_ev['variacao_absoluta']]
bars = ax.bar(df_ev['sigla_uf'], df_ev['variacao_absoluta'], color=cores)

# Valores nas barras
for bar, val in zip(bars, df_ev['variacao_absoluta']):
    ypos = val + 0.3 if val >= 0 else val - 0.8
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:+.1f}', ha='center', fontsize=8)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_ylabel('Variação (pontos percentuais)')
ax.set_title('Evolução da Taxa de Alfabetização 2023 → 2024', fontsize=14, fontweight='bold')

verde = mpatches.Patch(color='#2ecc71', label='Melhorou')
vermelho = mpatches.Patch(color='#e74c3c', label='Piorou')
ax.legend(handles=[verde, vermelho])
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4️⃣ Classificação dos Estados em 2024

In [ ]:
df_class = meta_vs_rea[meta_vs_rea['ano'] == 2024]['classificacao'].value_counts()

cores_pizza = ['#2ecc71', '#f39c12', '#e67e22', '#e74c3c']

fig, ax = plt.subplots(figsize=(8, 8))
wedges, texts, autotexts = ax.pie(
    df_class.values,
    labels=df_class.index,
    autopct='%1.0f%%',
    colors=cores_pizza[:len(df_class)],
    startangle=90
)
for text in autotexts:
    text.set_fontsize(12)
    text.set_fontweight('bold')

ax.set_title('Distribuição dos Estados por Classificação (2024)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nDetalhamento:')
print(df_class.to_string())

## 5️⃣ Painel Nacional

In [ ]:
print('=== PAINEL NACIONAL ===')
print(painel[['ano', 'taxa_media_nacional', 'media_portugues_nacional']].to_string(index=False))

if 'taxa_2023' in evolucao.columns and 'taxa_2024' in evolucao.columns:
    media_2023 = evolucao['taxa_2023'].mean()
    media_2024 = evolucao['taxa_2024'].mean()
    print(f'\nMédia nacional 2023: {media_2023:.1f}%')
    print(f'Média nacional 2024: {media_2024:.1f}%')
    print(f'Variação:            {media_2024 - media_2023:+.1f} pontos percentuais')
    print(f'Distância para meta 2030 (80%): {80 - media_2024:.1f} pontos')